# 04 · Bed change and window drift, one site

**The question.** In the n-tables (`01_fixed_config_n_tables_site.ipynb`) a window holds the n most recent
screened scenes on or before the 2025 Vaklodingen survey. Going from n = 40 to n = 60 adds older scenes, and
most methods get worse (centred RMSE, m):

| method | Westerschelde, n 40 → 60 | Wadden, n 40 → 60 |
|---|---|---|
| LogiTide2DEM (E4) | 0.357 → 0.368 | 0.224 → 0.254 |
| NIDEM | 0.345 → 0.361 | |
| DEA Intertidal | | 0.233 → 0.324 |
| sampled exceedance (E5c) | | 0.226 → 0.264 |

Two readings, to be told apart before paying for longer windows:

* **bed change (dataset shift):** the older looks saw an older bed;
* **season or sampling:** the added scenes fall in other months or other tide phases.

**What this notebook does.** Sections 1–3 measure how much the bed changes between the 2023, 2024 and 2025
Vaklodingen surveys at the production pixels. Sections 4–8 recompute MAREA, LogiTide2DEM and sampled exceedance
on the n-tables' own windows and pixels and ask whether the 40 → 60 degradation sits in the pixels that changed,
and whether the added scenes differ in season or tide phase. Sections 9–11 are a first try of time-aware
estimates for the two per-pixel methods where that is a small change (sampled exceedance and MAREA's
inversion): recency weights, and a refit after a detected break. These are tests of hypotheses: nothing of the
production method is changed.

**Inputs.** Those of the n-tables: the production extraction (pixels, NDWI, clear mask), the production
boundary, the screened scenes, the same windows, the same pixels F and the same functions. No field survey
other than the Vaklodingen is read.

**Outputs** (in `OUT`): `<site>_bed_change.csv`, `<site>_by_class.csv`, `<site>_shift.csv` (binned),
`<site>_shift_fit.csv`, `<site>_season.csv`, `<site>_time_aware.csv`, the per-pixel arrays `<site>_pixels.npz` and the figures.

## Parameters

Everything that changes between runs lives in the next cell. The environment variables `NB_SITE`, `NB_NS`,
`NB_OUT`, `NB_NTABLES` and `NB_YEARS` override the first five (the server script sets `NB_SITE`). The last
survey year of `SURVEY_YEARS` is the truth; the earlier ones measure the change.

In [ ]:
import os

SITE    = os.environ.get("NB_SITE", "escalda")                    # escalda | wadden | ems
NS      = os.environ.get("NB_NS")                                 # the short and the long window, e.g. "40,60";
                                                                  # unset: the site's pair (cell below)
OUT     = os.environ.get("NB_OUT", "results/bed_change_nb")       # the CSVs and figures go here
NTABLES = os.environ.get("NB_NTABLES", "results/fixedcfg_nb")     # the n-tables' CSVs: same windows, same scores
SURVEY_YEARS = [int(v) for v in os.environ.get("NB_YEARS", "2023,2024,2025").split(",")]

# ── The bed change ──────────────────────────────────────────────────────
CHANGE_STEPS_M = (0.1, 0.2, 0.5)    # the share of pixels changing by more than each
STABLE_M       = 0.1                # stable:   |z_2025 − z_y| < 0.1 m for every earlier survey y
CHANGING_M     = 0.2                # changing: |z_2025 − z_y| ≥ 0.2 m for at least one

# ── The time-aware estimates ────────────────────────────────────────────
RECENCY_T_YEARS = (0.5, 1.0, 2.0)   # (a) a look of age Δt weighs exp(−Δt / T)
CUSUM_CRIT      = 1.36              # (b) a break when max|CUSUM| / (sd √m) passes this (5 % of sup|Brownian bridge|)
MIN_AFTER       = 10                # (b) ... and at least this many valid looks follow it
YEAR_CHECK      = {"escalda": (2024, 2023)}   # windows ending 31 Dec of these years, scored against that survey

# ── Fixed by the n-tables ───────────────────────────────────────────────
MAX_PX            = 200_000         # a larger F is v19's seeded random subsample
SEED              = 20261019
RANGE_PAD_M       = 0.25            # truth kept inside the production no-delay range ± 0.25 m
MAREA_MIN_BRACKET = 0               # the rule of the published 10 m products (the n-tables' MAREA)

In [ ]:
# ── Each site as the n-tables run it ────────────────────────────────────
def dutch(site, n_gauges, judges, mouth_side, row_years, ns):
    d, tag = f"products_{site}_10m", f"eot20_g{n_gauges}"
    return dict(cache=f"ndwi_cube_{site}_2023-2025_10m.nc", extract=f"{d}/marea_extract.npz",
                overpass=f"{d}/overpass_times.json", marea=f"{d}/marea_{tag}",
                nodelay=(f"{d}/comparison_{tag}/products.npz", "z_plain"),
                period=("2023-01-01", "2025-12-31"), n_gauges=n_gauges, judges=judges,
                mouth_side=mouth_side, row_years=row_years, ns=ns)


SITES = {
    "escalda": dutch("escalda", 2, ["trnz"], "west", [2025], (40, 60)),
    "wadden":  dutch("wadden", 1, ["harl"], "north+west", [2025, 2024, 2023], (40, 60)),
    "ems":     dutch("ems", 1, ["delf"], "north+west", [2025], (40, 50)),    # 54 screened scenes: 60 is not reachable
}
P = SITES[SITE]
NS = [int(v) for v in NS.split(",")] if NS else list(P["ns"])
TRUTH_YEAR = SURVEY_YEARS[-1]
END = f"{TRUTH_YEAR}-12-31"                                              # the windows end there
ROW = f"{SITE}-{TRUTH_YEAR}" if len(P["row_years"]) > 1 else SITE        # the n-tables' row of these windows
print(f"{SITE}: windows n = {NS} ending {END}; surveys {SURVEY_YEARS}; the n-tables' row '{ROW}'")

## 0 · Setup

Every path is relative to the repository root; opened from `notebooks/validation/`, the first lines move there.
No Copernicus connection is needed: the cube, the extraction, the overpass times and the gauge records are cached.

In [ ]:
import sys
while not os.path.isdir("pyintertidal") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")                                    # from notebooks/validation/ to the repository root
sys.path.insert(0, os.getcwd())

import json, time, types, zlib
from collections import Counter
import numpy as np
import pandas as pd
import pyproj
import xarray as xr
import yaml
import matplotlib.pyplot as plt
from scipy import stats

import pyintertidal as pit
from pyintertidal import SentinelCube, marea
from pyintertidal import tide_estimators as te
from pyintertidal.boundary import make_boundary
from experiments.fixedcfg import methods as fm
from experiments.fixedcfg import elev_base, elev_freq
from experiments.v0_vaklodingen_fetch import sheets_for
from experiments.v1_vaklodingen_validate import sample_truth

pit.net.use_system_certificates()
os.makedirs(OUT, exist_ok=True)
print("repository", os.getcwd(), "· pyintertidal", pit.__version__)

## 1 · The production pixels and the surveys

The pixels are the production extraction's (`marea_extract.npz`, `keep`). Every pixel reads the 20 m Vaklodingen
cell under it with the n-tables' reader (`sample_truth` of `experiments/v1_vaklodingen_validate.py`: the nearest
cell, the first sheet with a value), one survey year at a time, so a year has a value only where a sheet was
surveyed that year. At the Westerschelde every sheet is surveyed every year; at the Wadden only some sheets are
surveyed more than once in 2023–2025; at the Ems none is. The year before each `YEAR_CHECK` year is read too
(section 11 classes its pixels by the change over that year).

In [ ]:
aoi = pit.sites.get(SITE)
cube = SentinelCube(aoi, P["period"], water="ndwi", cache_path=P["cache"], resolution=10)
cube.ensure(None)                                      # cached: no connection needed
transform, crs = cube.grid
H, W = cube.shape
ex = np.load(P["extract"])
keep = ex["keep"]                                      # flat indices of the production pixels in the H x W frame
print(f"{aoi} · grid {H}×{W} @ {abs(transform.a):g} m · {len(keep):,} production pixels")

In [ ]:
def vaklodingen(year):
    """{sheet: (x, y, z, year)} of the cells surveyed in `year` (a later survey of that year wins)."""
    cells = {}
    for name, _, _ in sheets_for(SITE):
        path = f"data_v4/truth/vaklodingen/{name}"
        if not os.path.exists(path):
            continue
        with xr.open_dataset(path) as ds:
            k = np.flatnonzero(ds["time"].dt.year.values == year)
            if not len(k):
                continue
            z, x, y = ds["z"].isel(time=k).values, ds["x"].values, ds["y"].values
        zt = np.full(z.shape[1:], np.nan)
        for zk in z:
            zt[np.isfinite(zk)] = zk[np.isfinite(zk)]
        cells[name] = (x, y, zt, np.full(zt.shape, year))
    return cells


with xr.open_dataset(P["cache"]) as ds:
    xs, ys = ds["x"].values, ds["y"].values
xq, yq = pyproj.Transformer.from_crs(crs, "EPSG:28992", always_xy=True).transform(xs[keep % W], ys[keep // W])

Z, SHEETS = {}, {}                                     # {year: z at every production pixel}, {year: sheets}
for year in sorted(set(SURVEY_YEARS) | {2023, 2024, 2025} | {y - 1 for y in YEAR_CHECK.get(SITE, ())}):   # 2023-2025 fix F
    cells = vaklodingen(year)
    Z[year], _ = sample_truth(cells, xq, yq)
    SHEETS[year] = sorted(cells)
    print(f"survey {year}: {len(cells):2d} sheet(s), {int(np.isfinite(Z[year]).sum()):>9,} of {len(keep):,} pixels with a value")
seen = Counter(s for y in SURVEY_YEARS for s in SHEETS[y])
print(f"sheets surveyed more than once in {SURVEY_YEARS}:", sorted(s for s, k in seen.items() if k > 1) or "none")

path, key = P["nodelay"]                               # the n-tables' range rule for the scored pixels
z_nodelay = np.load(path)[key]
lo, hi = np.nanmin(z_nodelay) - RANGE_PAD_M, np.nanmax(z_nodelay) + RANGE_PAD_M
print(f"the products' range ± {RANGE_PAD_M} m: [{lo:+.2f}, {hi:+.2f}] m")

## 2 · How much the bed changes

For every pair of surveys, Δz = z_later − z_earlier at the pixels where both have a value, on two sets: all the
production pixels, and those whose later survey lies inside the products' range ± 0.25 m (the n-tables' rule
for the scored pixels; it drops the channel cells a 20 m cell at the edge of a flat often falls in). The table
gives the distribution of Δz, its median absolute value, the share of pixels changing by more than 0.1, 0.2 and
0.5 m, and the share with Δz exactly 0.

In [ ]:
def change_stats(dz):
    """The distribution of the survey differences dz (m)."""
    a = np.abs(dz)
    out = dict(n_px=len(dz), median=np.median(dz), mean=dz.mean(), sd=dz.std(), p05=np.percentile(dz, 5),
               p95=np.percentile(dz, 95), median_abs=np.median(a), identical=np.mean(dz == 0))
    out.update({f"share_gt_{s:g}m": np.mean(a > s) for s in CHANGE_STEPS_M})
    return out


later = SURVEY_YEARS[-1]
PAIRS = [(later, y) for y in SURVEY_YEARS[-2::-1]] + list(zip(SURVEY_YEARS[1:-1], SURVEY_YEARS[:-2]))
rows = []
for b, a in PAIRS:
    both = np.isfinite(Z[b]) & np.isfinite(Z[a])
    for name, m in (("production pixels", both), ("inside the range", both & (Z[b] >= lo) & (Z[b] <= hi))):
        rows.append(dict(pair=f"{b} − {a}", pixels=name, **(change_stats(Z[b][m] - Z[a][m]) if m.any() else {"n_px": 0})))
bed = pd.DataFrame(rows)
bed.to_csv(f"{OUT}/{SITE}_bed_change.csv", index=False)
display(bed.round(3))

In [ ]:
show = [(b, a) for b, a in PAIRS if (np.isfinite(Z[b]) & np.isfinite(Z[a])).any()]
if show:
    fig, axes = plt.subplots(1, len(show), figsize=(5.2 * len(show), 3.6), squeeze=False)
    for ax, (b, a) in zip(axes[0], show):
        m = np.isfinite(Z[b]) & np.isfinite(Z[a]) & (Z[b] >= lo) & (Z[b] <= hi)
        dz = Z[b][m] - Z[a][m]
        ax.hist(np.clip(dz, -1.5, 1.5), bins=np.arange(-1.5, 1.52, 0.02), color="#2a78d6",
                weights=np.full(len(dz), 1 / max(len(dz), 1)))
        for v in (-CHANGING_M, -STABLE_M, STABLE_M, CHANGING_M):
            ax.axvline(v, color="0.35", lw=0.8, ls="--" if abs(v) == STABLE_M else ":")
        ax.set_title(f"z{b} − z{a}: {len(dz):,} px, {np.mean(np.abs(dz) >= CHANGING_M):.0%} ≥ {CHANGING_M} m",
                     loc="left", fontsize=10)
        ax.set_xlabel("Δz (m), clipped to ±1.5")
        ax.grid(alpha=0.3)
    axes[0][0].set_ylabel("share of pixels per 2 cm")
    fig.suptitle(f"{SITE}: survey differences inside the products' range", x=0.01, ha="left")
    fig.tight_layout()
    fig.savefig(f"{OUT}/{SITE}_bed_change_hist.png", dpi=130)
else:
    print(f"no pixel has two surveys among {SURVEY_YEARS}: the bed change cannot be measured here")

**How precise is a survey difference?** The Vaklodingen grids merge echo soundings (single-beam and multibeam,
from boats at high water: the channels and the lower flats) with airborne laser altimetry (LiDAR) of the dry
flats, interpolated to 20 m cells and dated by year only. Each source carries a vertical error of the order of
0.1 m, more on steep channel slopes (a 20 m cell averages the slope) and where the sources meet near the
low-water line; these files give no per-cell error. The survey month is unknown, so two surveys "one year apart"
may be 6 to 18 months apart, and the 2025 truth may predate the end of a window by months. **A difference below
about 0.1 m is within that noise**: hence *stable* is |Δz| < 0.1 m and *changing* is |Δz| ≥ 0.2 m, with a gap
between them. Both surveys are read at the same cell of the same grid, so no co-registration error enters Δz;
the four 10 m pixels of a cell share its value, so the shares are shares of pixels (≈ of area), not independent
cells. The `identical` column (Δz exactly 0) would show a survey that copies an older one.

## 3 · Where the change is

The maps show Δz between the last survey and each earlier one at the production pixels (red: the bed is higher
in the last survey). Every pixel then gets its **change class** from the largest |z_last − z_y| over the earlier
surveys y it has: *stable* (< 0.1 m in every one), *changing* (≥ 0.2 m in at least one), *between*, or
*no repeat survey*. The table gives the classes by elevation tercile of the last survey (inside the range), the
bridge to the season question: a sampling effect would act by elevation.

In [ ]:
zl = Z[later]
earlier = SURVEY_YEARS[:-1]
maps = [y for y in earlier if (np.isfinite(zl) & np.isfinite(Z[y])).any()]
if maps:
    fig, axes = plt.subplots(1, len(maps), figsize=(6.5 * len(maps), min(6.5 * H / W, 9)), squeeze=False)
    for ax, y in zip(axes[0], maps):
        grid = np.full(H * W, np.nan)
        grid[keep] = zl - Z[y]
        im = ax.imshow(grid.reshape(H, W), cmap="RdBu_r", vmin=-1, vmax=1, interpolation="nearest")
        ax.set_title(f"z{later} − z{y}", loc="left")
        ax.set_axis_off()
    fig.colorbar(im, ax=axes[0].tolist(), shrink=0.7, label="Δz (m); red: higher in the last survey")
    fig.savefig(f"{OUT}/{SITE}_bed_change_map.png", dpi=110)

dz_max = np.fmax.reduce(np.stack([np.abs(zl - Z[y]) for y in earlier]), axis=0)   # NaN: no earlier survey
change = np.where(np.isnan(dz_max), "no repeat survey",
                  np.where(dz_max < STABLE_M, "stable", np.where(dz_max >= CHANGING_M, "changing", "between")))
n_old = np.sum([np.isfinite(Z[y]) for y in earlier], axis=0)
z_old = np.sum([np.nan_to_num(Z[y]) for y in earlier], axis=0) / np.where(n_old > 0, n_old, np.nan)   # the older bed
CLASSES = ["stable", "between", "changing", "no repeat survey"]
inr = np.isfinite(zl) & (zl >= lo) & (zl <= hi)
print("change class of the production pixels inside the range:",
      {c: f"{np.mean(change[inr] == c):.1%}" for c in CLASSES})
rep = inr & (change != "no repeat survey")
if rep.sum() >= 30:
    display(pd.crosstab(pd.qcut(zl[rep], 3), change[rep], normalize="index").reindex(columns=CLASSES[:3]).round(3))

## 4 · The screened scenes, the windows and the water levels

The n-tables' screened list S at these sites is the production MAREA record (`record_dates` of its
`result.json`): the n-tables notebook checked it at all three Dutch sites, where no pre-2023 scenes were added.
This notebook reads S from the record instead of rerunning the screening (4 to 55 minutes a site), with the
overpass times and the production boundary (EOT20 and the nearest gauges, the judges held out). Window(n) is the
n most recent images of S on or before the truth date; each window is checked against the n-tables' CSV (same
first and last image; the windows are nested in S, so n and the two ends fix the images).

The levels h_B(t − τ) are read once for every scene of S and every whole minute τ from −45 to 180 (MAREA's lag
bank and its per-pixel lags; τ = 0 is what the other methods read), in one boundary call, as in the n-tables.

In [ ]:
record = [d[:10] for d in json.load(open(f"{P['marea']}/result.json"))["record_dates"]]
overpass = {d: pd.Timestamp(v) for d, v in json.load(open(P["overpass"])).items()}
S_dates = np.array(record)
S_t = pd.DatetimeIndex([overpass[d] for d in S_dates]).tz_localize(None)
boundary = make_boundary(aoi, tide_model="EOT20", n_gauges=P["n_gauges"], period=P["period"],
                         exclude=P["judges"])
print(f"boundary: {boundary.name}")
print(f"S: {len(S_dates)} screened scenes {S_dates[0]} → {S_dates[-1]}; every one with a finite level: "
      f"{bool(np.isfinite(np.asarray(boundary.levels(S_t), float)).all())}")


def window(end, n):
    """The n most recent images of S on or before `end` (positions in S), or None when S has fewer."""
    upto = np.flatnonzero(S_dates <= end)
    return upto[-n:] if len(upto) >= n else None


csv = f"{NTABLES}/{SITE}.csv"
nt = pd.read_csv(csv) if os.path.exists(csv) else None
WIN = {}
for n in NS:
    w = window(END, n)
    if w is None:
        print(f"n={n:3d}: not reachable, {int((S_dates <= END).sum())} screened images on or before {END}")
        continue
    WIN[n] = w
    d = S_dates[w]
    same = "no n-tables CSV"
    if nt is not None:
        r = nt[(nt.site == ROW) & (nt.n == n) & nt.first_date.notna()]
        same = (f"the n-tables' window: {(r.first_date.iloc[0], r.last_date.iloc[0]) == (d[0], d[-1])}"
                if len(r) else "not in the n-tables")
    print(f"n={n:3d}: {d[0]} → {d[-1]}  images per year {dict(sorted(Counter(x[:4] for x in d).items()))} · {same}")
assert len(WIN) == 2, "both windows must be reachable"
N_SHORT, N_LONG = sorted(WIN)

In [ ]:
shifted = {tau: S_t - pd.Timedelta(minutes=tau) for tau in range(-45, 181)}
inst = pd.DatetimeIndex(np.unique(np.concatenate([s.values for s in shifted.values()])))
lev = np.asarray(boundary.levels(inst), float)
LEV = {tau: lev[np.searchsorted(inst.values, s.values)] for tau, s in shifted.items()}
print(f"h_B(t - tau) at {len(S_t)} scenes, tau = -45..180 min; range at tau = 0 [{LEV[0].min():+.2f}, {LEV[0].max():+.2f}] m")

pos = {d: i for i, d in enumerate(ex["dates"].astype(str))}
rows_S = [pos[d] for d in S_dates]
Y_S, C_S = ex["Y"][rows_S], ex["C"][rows_S]             # (scenes of S, pixels): NDWI and clear
print(f"Y and C of S: {Y_S.shape[0]} scenes × {Y_S.shape[1]:,} pixels")

seeds = pit.geometry.mouth_seeds(ex["sea"], side=P["mouth_side"])
s_km = pit.geometry.along_distance(ex["sea"] | ex["inter"], seeds, 10).ravel()[keep] / 1000.0
BAND_KM = 3.0
_, centers, band_of = te.make_bands_fixed(s_km, BAND_KM, 500)
production = np.load(f"{P['marea']}/marea.npz")
print(f"{len(centers)} bands of {BAND_KM:g} km; the production product's bands: {np.array_equal(band_of, production['band'])}")

## 5 · The pixels F and their change class

F exactly as the n-tables build it: the most recent of the 2025, 2024 and 2023 surveys with a value gives each
pixel its truth and survey year; the truth inside the products' range ± 0.25 m; the survey years of the n-tables'
rows; above 200,000 pixels v19's seeded subsample. The windows of this notebook are the row of the truth year, so
its pixels are F's pixels of that year (the n-tables' `n_F`). Each carries the change class of section 3.

In [ ]:
truth, group = np.full(len(keep), np.nan), np.zeros(len(keep), int)
for year in (2025, 2024, 2023):                        # the most recent survey with a value wins
    new = np.isfinite(Z[year]) & (group == 0)
    truth[new], group[new] = Z[year][new], year
in_range = (truth >= lo) & (truth <= hi)
F = np.flatnonzero(in_range & np.isin(group, P["row_years"]))
if len(F) > MAX_PX:                                    # v19's seeded subsample: the n-tables' pixels
    rng = np.random.default_rng(np.random.SeedSequence([SEED, zlib.crc32(SITE.encode()) & 0x7FFFFFFF, 7]))
    F = F[np.sort(rng.permutation(len(F))[:MAX_PX])]
px = F[group[F] == TRUTH_YEAR]                         # positions in keep
t_true, cls = truth[px], change[px]
n_F = nt[(nt.site == ROW) & nt.n_F.notna()].n_F.iloc[0] if nt is not None else np.nan
print(f"F: {len(F):,} pixels; the {TRUTH_YEAR} row: {len(px):,} pixels (the n-tables' n_F: {n_F:,.0f})")
display(pd.Series(cls).value_counts().reindex(CLASSES).fillna(0).astype(int).to_frame("pixels").T)

## 6 · The three methods

**MAREA** as the n-tables run it, in two calls so that the time-aware variants (section 9) reuse the window's
lags: `marea_lags` refits the lag of every 3-km band on the window's images (`te.m2a_rasch`, production's
settings from `configs/m2.yaml`, its seeded draw), sets a negative band lag to 0, smooths the profile along the
mouth distance and rounds it to the minute; `marea_invert` inverts every pixel with the level read τ minutes
earlier (`marea.invert_series`), between the window's lowest and highest level. Its `clear` argument replaces
the clear mask by a weight per look (section 9). **LogiTide2DEM (E4)** and **sampled exceedance (E5c)** are
`run(obs, cfg)` of `experiments/fixedcfg/methods.py` on the common inputs, as the n-tables call them.
The scores are the n-tables' `scores`.

In [ ]:
M2 = yaml.safe_load(open("configs/m2.yaml", encoding="utf-8"))
FIT_KW = dict(sigma0=M2["sigma0_m"], sg_grid=tuple(M2["sigma_perfil_m"]), tau_bounds=marea.TAU_BOUNDS,
              z_points=M2["z_puntos"], max_px_band=M2["max_px_banda"]["m2a"], monotone=True)


def marea_lags(w):
    """The band lags (min) fitted on the images w, and every production pixel's lag (whole minutes)."""
    bank = te.ShiftBank(marea.BANK_TAUS, [LEV[int(tau)][w] for tau in marea.BANK_TAUS])
    with np.errstate(invalid="ignore"):
        wet = C_S[w] & (Y_S[w] > 0)
    fit = te.m2a_rasch(wet, C_S[w], bank, band_of, centers, len(centers),
                       rng=np.random.default_rng(M2["seed"] + 31), **FIT_KW)
    tau_bands = np.clip(np.asarray(fit["tau"], float), 0, None)       # causality: a negative band lag is 0
    tau_px = te.smooth_lag_profile(centers, tau_bands, s_km, bandwidth_km=BAND_KM)
    return tau_bands, np.where(np.isfinite(tau_px), np.round(tau_px), 0.0)


def marea_invert(w, px, tau_px, clear=None):
    """MAREA's elevation at the pixels px (positions in keep) from the images w; `clear` (images x px) replaces
    the clear mask, as a weight per look (0 = not used)."""
    h0 = LEV[0][w]
    Y0 = np.nan_to_num(Y_S[np.ix_(w, px)], nan=0.0)
    C0 = C_S[np.ix_(w, px)] if clear is None else clear
    tau = tau_px[px]
    z = np.full(len(px), np.nan)
    for t in np.unique(tau):
        cols = np.flatnonzero(tau == t)
        z[cols], _ = marea.invert_series(Y0[:, cols], C0[:, cols], LEV[int(t)][w], h0.min(), h0.max(),
                                         min_bracket=MAREA_MIN_BRACKET)
    return z


def make_obs(w, px):
    """The common inputs of the images w at the pixels px."""
    Y, C = Y_S[np.ix_(w, px)], C_S[np.ix_(w, px)]
    with np.errstate(invalid="ignore"):
        wet = C & (Y > 0)                              # an undefined NDWI is never wet
    return types.SimpleNamespace(Y=Y, C=C, W=wet, h=LEV[0][w], t=S_t[w], keep=keep[px],
                                 shape=(H, W), pixel_m=10.0, site=SITE)


def run_competitor(mid, w, px):
    return np.asarray(fm.METHODS[mid][1](make_obs(w, px), {})["z"], float)


def scores(p, t):
    """RMSE, bias, centred RMSE, slope, r, the pixels scored and the coverage of p against t (same pixels)."""
    p, t = np.asarray(p, float), np.asarray(t, float)
    ok = np.isfinite(p) & np.isfinite(t)
    out = dict(n_px=int(ok.sum()), coverage=float(ok.mean()) if len(ok) else np.nan,
               rmse=np.nan, bias=np.nan, crmse=np.nan, slope=np.nan, r=np.nan)
    if ok.sum() >= 30:
        d = p[ok] - t[ok]
        pc, tc = p[ok] - np.median(p[ok]), t[ok] - np.median(t[ok])
        out.update(rmse=float(np.sqrt(np.mean(d ** 2))), bias=float(np.median(d)),
                   crmse=float(np.sqrt(np.mean((pc - tc) ** 2))), slope=float(np.polyfit(tc, pc, 1)[0]),
                   r=float(np.corrcoef(tc, pc)[0, 1]))
    return out


METHODS = ["MAREA", "E4", "E5c"]
LABELS = {"MAREA": "MAREA", "E4": "LogiTide2DEM", "E5c": "sampled exceedance"}
COLORS = {"MAREA": "#0b0b0b", "E4": "#1baf7a", "E5c": "#008300"}

## 7 · n = 40 against n = 60: where does the error grow?

Both windows, the three methods, on the truth year's pixels of F. The first table checks the scores against the
n-tables' CSV (the same windows, pixels and functions give the same numbers).

Then, per method, on the pixels where both windows and the truth have a value, the centred error
e = (p − median p) − (t − median t) of each window, and per change class: the centred RMSE of each window
within the class (the n-tables' score on that class), the change in mean squared error within the class,
d_mse = mean(e_long² − e_short²), and its contribution to the whole, share_px × d_mse (the contributions add up
to the `all` row's d_mse). **If the degradation sits in the changing pixels, the drift is real.**

In [ ]:
EST, LAGS = {}, {}
for n, w in WIN.items():
    t0 = time.perf_counter()
    tau_bands, LAGS[n] = marea_lags(w)
    EST[n, "MAREA"] = marea_invert(w, px, LAGS[n])
    for mid in ("E4", "E5c"):
        EST[n, mid] = run_competitor(mid, w, px)
    print(f"n={n}: MAREA band lags {'/'.join(f'{v:.0f}' for v in tau_bands)} min  [{time.perf_counter() - t0:.0f} s]")

check = []
for (n, mid), z in sorted(EST.items()):
    s = scores(z, t_true)
    ref = nt[(nt.site == ROW) & (nt.n == n) & (nt.method == mid)] if nt is not None else pd.DataFrame()
    check.append(dict(method=mid, n=n, crmse=s["crmse"], crmse_ntables=ref.crmse.iloc[0] if len(ref) else np.nan,
                      coverage=s["coverage"], coverage_ntables=ref.coverage.iloc[0] if len(ref) else np.nan))
check = pd.DataFrame(check)
display(check.round(4))
print("max |crmse − n-tables|:", float(np.nanmax(np.abs(check.crmse - check.crmse_ntables))))

In [ ]:
def centred_errors(mid):
    """The pixels g where both windows and the truth have a value, and each window's centred error there."""
    a, b = EST[N_SHORT, mid], EST[N_LONG, mid]
    g = np.isfinite(a) & np.isfinite(b) & np.isfinite(t_true)
    tc = t_true[g] - np.median(t_true[g])
    return g, a[g] - np.median(a[g]) - tc, b[g] - np.median(b[g]) - tc


rows = []
for mid in METHODS:
    g, ea, eb = centred_errors(mid)
    for c in CLASSES + ["all"]:
        m = cls[g] == c if c != "all" else np.ones(g.sum(), bool)
        if not m.any():
            continue
        d_mse = np.mean(eb[m] ** 2 - ea[m] ** 2)
        rows.append(dict(method=mid, change_class=c, n_px=int(m.sum()), share_px=m.mean(),
                         crmse_short=scores(EST[N_SHORT, mid][g][m], t_true[g][m])["crmse"],
                         crmse_long=scores(EST[N_LONG, mid][g][m], t_true[g][m])["crmse"],
                         d_mse=d_mse, contribution=m.mean() * d_mse))
by_class = pd.DataFrame(rows)
by_class["crmse_change"] = by_class.crmse_long - by_class.crmse_short
by_class.to_csv(f"{OUT}/{SITE}_by_class.csv", index=False)
print(f"centred RMSE, n = {N_SHORT} → {N_LONG}, by change class (m):")
display(by_class.round(3))

**The shift test.** If the added looks saw an older bed, the long window's estimate moves from the short one's
toward that bed: ẑ_long − ẑ_short grows with z_old − z_last (z_old: the mean of the earlier surveys at the pixel).
A frequency method would move by roughly the added share of looks, (n_long − n_short)/n_long, of that difference
(the dashed line); without drift the slope is 0. The elevation z_last enters as a second regressor, so that a
sampling effect that depends on elevation is not read as drift. Survey noise in z_old − z_last pulls the slope
toward 0.

In [ ]:
BINS = [-np.inf, -0.5, -0.2, -0.1, 0.1, 0.2, 0.5, np.inf]
z_old_px = z_old[px]
rows, binned = [], []
for mid in METHODS:
    a, b = EST[N_SHORT, mid], EST[N_LONG, mid]
    g = np.isfinite(a) & np.isfinite(b) & np.isfinite(t_true) & np.isfinite(z_old_px)
    if g.sum() < 30:
        print(f"{mid}: {int(g.sum())} pixels with an earlier survey: no shift test")
        continue
    shift, x, elev = b[g] - a[g], z_old_px[g] - t_true[g], t_true[g]
    coef = np.linalg.lstsq(np.column_stack([np.ones(len(x)), x, elev]), shift, rcond=None)[0]
    rows.append(dict(method=mid, n_px=int(g.sum()), mean_shift=shift.mean(), slope=np.polyfit(x, shift, 1)[0],
                     slope_given_elevation=coef[1], elevation_coef=coef[2], r=np.corrcoef(x, shift)[0, 1]))
    d = pd.DataFrame(dict(bin=pd.cut(x, BINS), x=x, shift=shift))
    binned.append(d.groupby("bin", observed=True).agg(x=("x", "median"), shift=("shift", "mean"), n_px=("x", "size"))
                  .reset_index().assign(method=mid))
shift_tab = pd.DataFrame(rows)
shift_tab.to_csv(f"{OUT}/{SITE}_shift_fit.csv", index=False)
display(shift_tab.round(3))
if binned:
    binned = pd.concat(binned)
    binned.to_csv(f"{OUT}/{SITE}_shift.csv", index=False)
    display(binned.pivot_table(index="bin", columns="method", values="shift", observed=True).round(3))

In [ ]:
fig, axes = plt.subplots(1, len(METHODS) + 1, figsize=(4.2 * (len(METHODS) + 1), 3.8))
for ax, mid in zip(axes, METHODS):
    d = by_class[(by_class.method == mid) & by_class.change_class.isin(CLASSES)]
    xk = np.arange(len(d))
    ax.bar(xk - 0.2, d.crmse_short, 0.4, color="#2a78d6", label=f"n = {N_SHORT}")
    ax.bar(xk + 0.2, d.crmse_long, 0.4, color="#eb6834", label=f"n = {N_LONG}")
    ax.set_xticks(xk, [f"{c}\n{k:,} px" for c, k in zip(d.change_class, d.n_px)], fontsize=8)
    ax.set_title(f"{LABELS[mid]}: centred RMSE (m)", loc="left", fontsize=10)
    ax.grid(axis="y", alpha=0.3)
axes[0].legend(fontsize=8)
ax = axes[-1]
if len(binned):
    for mid, d in binned.groupby("method"):
        ax.plot(d.x, d["shift"], marker="o", color=COLORS[mid], label=LABELS[mid])
    xx = np.array([binned.x.min(), binned.x.max()])
    ax.plot(xx, xx * (N_LONG - N_SHORT) / N_LONG, color="0.5", ls="--", lw=1, label="added share × Δ")
    ax.axhline(0, color="0.5", lw=0.8)
    ax.legend(fontsize=8)
ax.set_title(f"mean ẑ{N_LONG} − ẑ{N_SHORT} (m)", loc="left", fontsize=10)
ax.set_xlabel("z_old − z_last (m), bin median")
ax.grid(alpha=0.3)
fig.suptitle(f"{SITE}: the added older images, by measured bed change", x=0.01, ha="left")
fig.tight_layout()
fig.savefig(f"{OUT}/{SITE}_by_class.png", dpi=130)

## 8 · Season and tide phase of the added scenes

The added scenes are the long window's oldest n_long − n_short. Their months, their boundary levels (τ = 0),
their tide phase (rising when the level 30 min later is above the level 30 min earlier) and their clear share at
the scored pixels are set beside the short window's; the levels are compared with a two-sample
Kolmogorov–Smirnov test. A sampling difference would add error by elevation (missed low or high levels) in stable
and changing pixels alike: the last table splits the change in centred RMS error by elevation tercile within the
stable and the changing pixels.

In [ ]:
w_long, w_short = WIN[N_LONG], WIN[N_SHORT]
added = w_long[:len(w_long) - len(w_short)]
assert np.array_equal(w_long[len(added):], w_short), "the windows are nested"
SETS = {f"window n={N_SHORT}": w_short, f"added {len(added)} (older)": added}
rows = []
for name, w in SETS.items():
    h = LEV[0][w]
    rows.append(dict(scenes=name, n=len(w), first=S_dates[w][0], last=S_dates[w][-1], h_min=h.min(),
                     h_p10=np.percentile(h, 10), h_median=np.median(h), h_p90=np.percentile(h, 90), h_max=h.max(),
                     rising_share=np.mean(LEV[-30][w] > LEV[30][w]), clear_share_at_px=C_S[np.ix_(w, px)].mean()))
season = pd.DataFrame(rows)
season.to_csv(f"{OUT}/{SITE}_season.csv", index=False)
display(season.round(3))
ks = stats.ks_2samp(LEV[0][w_short], LEV[0][added])
print(f"levels, added vs the short window: KS D = {ks.statistic:.2f}, p = {ks.pvalue:.2f}")
months = pd.DataFrame({k: pd.Series(Counter(int(d[5:7]) for d in S_dates[w])) for k, w in SETS.items()})
display(months.reindex(range(1, 13)).fillna(0).astype(int).T)

rows = []
for mid in METHODS:
    g, ea, eb = centred_errors(mid)
    terc = np.asarray(pd.qcut(t_true[g], 3, labels=["low", "mid", "high"]))
    for c in ("stable", "changing", "all"):
        for k in ("low", "mid", "high"):
            m = (terc == k) & ((cls[g] == c) if c != "all" else True)
            if m.sum() >= 30:
                rows.append(dict(method=mid, change_class=c, elevation=k, n_px=int(m.sum()),
                                 rms_change=np.sqrt(np.mean(eb[m] ** 2)) - np.sqrt(np.mean(ea[m] ** 2))))
if rows:
    print(f"change in centred RMS error n = {N_SHORT} → {N_LONG} (m), by elevation tercile:")
    display(pd.DataFrame(rows).pivot_table(index=["method", "change_class"], columns="elevation",
                                           values="rms_change").reindex(columns=["low", "mid", "high"]).round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for (name, w), color, off in zip(SETS.items(), ("#2a78d6", "#eb6834"), (-0.2, 0.2)):
    mm = Counter(int(d[5:7]) for d in S_dates[w])
    axes[0].bar(np.arange(1, 13) + off, [mm.get(k, 0) for k in range(1, 13)], 0.4, color=color, label=name)
    axes[1].hist(LEV[0][w], bins=np.linspace(LEV[0].min(), LEV[0].max(), 16), histtype="step", lw=2,
                 color=color, density=True, label=name)
axes[0].set_xticks(range(1, 13))
axes[0].set_title("images per month", loc="left")
axes[1].set_title("boundary level at the image (m)", loc="left")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(f"{OUT}/{SITE}_season.png", dpi=130)

## 9 · Time-aware estimates: two first tries

Only for the two per-pixel methods where it is a small change; the band lags of MAREA stay those of the static
fit of the window.

**(a) Recency weights.** A look of age Δt (before the window's last image) weighs w = exp(−Δt/T), T = 0.5, 1
and 2 years.
*Sampled exceedance:* f_w = the weighted wet share of the pixel's valid looks, and z = the weighted (1 − f_w)
quantile of their levels: each sorted look sits at the middle of its share of the weight, z is interpolated
between them and clamped to the lowest and highest level. In a step world z stays between the highest dry and
the lowest wet level whatever the weights. With equal weights this is the Hazen quantile, a quarter of a level
step at most from E5c's linear one, so its own equal-weight row (T = ∞) is the static reference of the weighted
rows (checked below).
*MAREA:* `marea.invert_series` is a least-squares fit that takes a weight per look in place of the clear mask; the
weights are rescaled per pixel to sum to its number of clear looks, which leaves the fit's minimum unchanged and
keeps its guard (at least 8 looks) counting looks. Weights all 1 give the static fit exactly (checked below).

**(b) After the last change.** Per pixel and in time order, the residual of each valid look against the
method's fitted curve, here the step at its own static estimate: r = wet − [h > ẑ] (at the levels the method
reads: τ = 0 for E5c, the pixel's lag for MAREA). The CUSUM of the centred residuals has its break at its largest
|value|; the break is taken when max|S| / (sd √m) > 1.36 (5 % of the supremum of a Brownian bridge) and at least
10 valid looks follow it. The method is then refitted on the looks after the break (E5c with its own
`sampled_exceedance_block`); a pixel without a break keeps all its looks. Over tens of thousands of pixels some
breaks are false alarms, which only cost looks; the share of pixels with a break, stable against changing, shows
whether the detector sees real change. The last cell of this section measures the detector's power on the long
window's own levels, with synthetic pixels whose bed was 0.2, 0.5 or 1 m higher before the short window starts:
a look tells about the bed only when its level falls between the old and the new bed, so a small change leaves
few looks to see it.

In [ ]:
def recency(w, T_years):
    """exp(−Δt / T) for the images w, Δt in years before the window's last image."""
    age = np.asarray((S_t[w][-1] - S_t[w]).total_seconds(), float) / (365.25 * 86400)
    return np.exp(-age / T_years)


def weighted_exceedance(V, Wb, h, w):
    """Sampled exceedance with a weight per look (w: looks x pixels, or looks x 1; 0 drops the look)."""
    V = V & (w > 0)
    wv = np.where(V, w, 0.0)
    tot = wv.sum(0)
    with np.errstate(invalid="ignore", divide="ignore"):
        q = 1.0 - (wv * Wb).sum(0) / tot                       # the weighted dry share
    order = np.argsort(np.where(V, h[:, None], np.inf), axis=0, kind="stable")
    hs = np.take_along_axis(np.broadcast_to(h[:, None], V.shape), order, 0)   # valid levels first, ascending
    ws = np.take_along_axis(wv, order, 0)
    with np.errstate(invalid="ignore", divide="ignore"):
        x = (np.cumsum(ws, 0) - ws / 2) / tot                  # each look at the middle of its weight
    m, cols = V.sum(0), np.arange(V.shape[1])
    up = np.minimum(((x < q) & (np.arange(len(h))[:, None] < m)).sum(0), np.maximum(m - 1, 0))
    dn = np.maximum(up - 1, 0)                                 # the looks either side of q
    xd, xu = x[dn, cols], x[up, cols]
    with np.errstate(invalid="ignore", divide="ignore"):
        frac = np.clip(np.where(xu > xd, (q - xd) / (xu - xd), 1.0), 0, 1)
    z = hs[dn, cols] + frac * (hs[up, cols] - hs[dn, cols])
    return np.where(m > 0, z, np.nan)


def cusum_break(V, Wb, h, z):
    """Per pixel, the last look before a break in the wet/dry-versus-level relation (-1: no break), and the
    statistic max|S| / (sd √m). h: looks x pixels (or looks x 1), z: the pixel's static estimate."""
    e = np.where(V, Wb.astype(float) - (h > z[None, :]), 0.0)    # the residual of the fitted step
    m, cols = V.sum(0), np.arange(V.shape[1])
    ec = np.where(V, e - e.sum(0) / np.maximum(m, 1), 0.0)
    S = np.cumsum(ec, 0)
    k = np.abs(S).argmax(0)
    with np.errstate(invalid="ignore", divide="ignore"):
        stat = np.abs(S[k, cols]) / (np.sqrt((ec ** 2).sum(0) / np.maximum(m, 1)) * np.sqrt(m))
    after = m - np.cumsum(V, 0)[k, cols]
    found = (stat > CUSUM_CRIT) & (after >= MIN_AFTER) & np.isfinite(z)
    return np.where(found, k, -1), stat


def e5c_variants(w, px):
    """Sampled exceedance on the images w: as registered, with equal and recency weights, after the last change."""
    o = make_obs(w, px)
    V, Wb = elev_base.valid_and_wet(o.Y, o.C, o.W, slice(None))
    z0 = run_competitor("E5c", w, px)
    out = {"static": z0, "weights T=inf": weighted_exceedance(V, Wb, o.h, np.ones((len(w), 1)))}
    for T in RECENCY_T_YEARS:
        out[f"recency T={T:g} y"] = weighted_exceedance(V, Wb, o.h, recency(w, T)[:, None])
    k, _ = cusum_break(V, Wb, o.h[:, None], z0)
    after = np.arange(len(w))[:, None] > k[None, :]
    out["after the last change"] = elev_freq.sampled_exceedance_block(V & after, Wb & after, o.h)
    return out, k


def marea_variants(w, px, tau_px):
    """MAREA's inversion on the images w (the window's band lags): static, recency weights, after the last change."""
    C0 = C_S[np.ix_(w, px)]
    z0 = marea_invert(w, px, tau_px)
    out = {"static": z0}
    for T in RECENCY_T_YEARS:
        cw = C0 * recency(w, T)[:, None]
        out[f"recency T={T:g} y"] = marea_invert(w, px, tau_px, clear=cw * (C0.sum(0) / np.maximum(cw.sum(0), 1e-12)))
    h = np.empty(C0.shape)
    tau = tau_px[px]
    for t in np.unique(tau):
        h[:, tau == t] = LEV[int(t)][w][:, None]
    with np.errstate(invalid="ignore"):
        wet = C0 & (Y_S[np.ix_(w, px)] > 0)
    k, _ = cusum_break(C0, wet, h, z0)
    out["after the last change"] = marea_invert(w, px, tau_px, clear=C0 & (np.arange(len(w))[:, None] > k[None, :]))
    return out, k


w = WIN[N_SHORT]
o = make_obs(w, px)
V, Wb = elev_base.valid_and_wet(o.Y, o.C, o.W, slice(None))
zw = weighted_exceedance(V, Wb, o.h, np.ones((len(w), 1)))
z0 = EST[N_SHORT, "E5c"]
print(f"E5c, equal weights against as registered (n={N_SHORT}): same NaN pattern {np.array_equal(np.isnan(zw), np.isnan(z0))}, "
      f"max |diff| {np.nanmax(np.abs(zw - z0)):.3f} m, median |diff| {np.nanmedian(np.abs(zw - z0)):.4f} m")
zm = marea_invert(w, px, LAGS[N_SHORT], clear=C_S[np.ix_(w, px)].astype(float))
print(f"MAREA, weights all 1 against static (n={N_SHORT}): identical {np.array_equal(zm, EST[N_SHORT, 'MAREA'], equal_nan=True)}")

In [ ]:
# the detector's power on the long window's own levels: synthetic pixels whose bed is `shift` higher before the
# short window starts (the window's clear share, 2 % of the looks misclassified)
w = WIN[N_LONG]
h, cut = LEV[0][w], len(WIN[N_LONG]) - N_SHORT
clear = C_S[np.ix_(w, px)].mean()
rng = np.random.default_rng(0)
for shift in (0.0, 0.2, 0.5, 1.0):
    bed = rng.uniform(np.percentile(h, 10), np.percentile(h, 90), 2000)
    beds = np.where(np.arange(len(w))[:, None] < cut, bed + shift, bed)
    V = rng.random((len(w), 2000)) < clear
    Wb = (V & (h[:, None] > beds)) ^ (V & (rng.random(V.shape) < 0.02))
    k, _ = cusum_break(V, Wb, h[:, None], elev_freq.sampled_exceedance_block(V, Wb, h))
    print(f"bed {shift:.1f} m higher before {S_dates[w][cut]}: a break in {np.mean(k >= 0):5.1%} of 2,000 synthetic "
          f"pixels, at the right image ±3 in {np.mean(np.abs(k[k >= 0] - (cut - 1)) <= 3) if (k >= 0).any() else np.nan:.0%}")

## 10 · The time-aware estimates on the long window, against the last survey

The long window (n = 60 at the Westerschelde and the Wadden), every variant at the truth year's pixels of F,
scored on the pixels where every variant of the method and the truth have a value: overall, stable and changing.
The short window's static estimate is the reference row (what dropping the old images gives). Then the share of
pixels with a detected break, by change class, and the year of the last look before the break.

In [ ]:
def score_variants(variants, t, c):
    """Centred RMSE of every variant, overall and by change class, on the pixels where all have a value."""
    g = np.isfinite(t)
    for z in variants.values():
        g &= np.isfinite(z)
    rows = []
    for name, z in variants.items():
        r = dict(variant=name, coverage=float(np.isfinite(z).mean()))
        for k in ("all", "stable", "changing"):
            m = g & ((c == k) if k != "all" else True)
            r[f"n_{k}"], r[f"crmse_{k}"] = int(m.sum()), scores(z[m], t[m])["crmse"]
        rows.append(r)
    return rows


w = WIN[N_LONG]
TA, BREAKS = {}, {}
t0 = time.perf_counter()
TA["E5c"], BREAKS["E5c"] = e5c_variants(w, px)
TA["MAREA"], BREAKS["MAREA"] = marea_variants(w, px, LAGS[N_LONG])
print(f"variants on n={N_LONG}  [{time.perf_counter() - t0:.0f} s]")
rows = []
for mid, variants in TA.items():
    named = {f"static n={N_SHORT}": EST[N_SHORT, mid]}
    named.update({(f"{k} n={N_LONG}" if k.startswith("static") else k): v for k, v in variants.items()})
    rows += [dict(window=f"n={N_LONG} → {TRUTH_YEAR}", method=mid, **r) for r in score_variants(named, t_true, cls)]
time_aware = pd.DataFrame(rows)
display(time_aware.round(3))

for mid, k in BREAKS.items():
    print(f"{mid}: share of pixels with a break", {c: f"{np.mean(k[cls == c] >= 0):.1%}" for c in CLASSES if (cls == c).any()},
          "· last look before it, by year", dict(sorted(Counter(str(d)[:4] for d in S_dates[w][k[k >= 0]]).items())))

In [ ]:
fig, axes = plt.subplots(1, len(TA), figsize=(6.4 * len(TA), 3.8), squeeze=False)
for ax, (mid, d) in zip(axes[0], time_aware.groupby("method", sort=False)):
    xk = np.arange(len(d))
    ax.bar(xk - 0.2, d.crmse_stable, 0.4, color="#2a78d6", label="stable")
    ax.bar(xk + 0.2, d.crmse_changing, 0.4, color="#eb6834", label="changing")
    ax.set_xticks(xk, d.variant, rotation=30, ha="right", fontsize=8)
    ax.set_title(f"{LABELS[mid]}: centred RMSE (m) against {TRUTH_YEAR}", loc="left", fontsize=10)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(f"{OUT}/{SITE}_time_aware.png", dpi=130)

## 11 · Does the time-aware fit track each year? (Westerschelde)

Where every sheet is surveyed every year, a window ending 31 December 2024 is scored against the 2024 survey and
one ending 31 December 2023 against the 2023 survey, both from the 2023–2025 cube: the n_long most recent images,
or all of them on or before the end when the cube holds fewer. The pixels are the truth year's pixels of F with a
value of that year inside the range; the change class is the change over the year before (|z_y − z_(y−1)|). The
band lags are refitted on each window.

In [ ]:
rows = []
for year in YEAR_CHECK.get(SITE, ()):
    end = f"{year}-12-31"
    n = min(N_LONG, int((S_dates <= end).sum()))
    w = window(end, n)
    zy = Z[year]
    pxy = px[np.isfinite(zy[px]) & (zy[px] >= lo) & (zy[px] <= hi)]
    dy = np.abs(zy[pxy] - Z[year - 1][pxy])
    cy = np.where(np.isnan(dy), "no repeat survey",
                  np.where(dy < STABLE_M, "stable", np.where(dy >= CHANGING_M, "changing", "between")))
    print(f"{year}: n={n}, {S_dates[w][0]} → {S_dates[w][-1]}, {len(pxy):,} pixels "
          f"({np.mean(cy == 'stable'):.0%} stable, {np.mean(cy == 'changing'):.0%} changing over {year - 1}–{year})")
    tau_bands, tau_px = marea_lags(w)
    for mid, (variants, _) in (("E5c", e5c_variants(w, pxy)), ("MAREA", marea_variants(w, pxy, tau_px))):
        named = {(f"{k} n={n}" if k.startswith("static") else k): v for k, v in variants.items()}
        rows += [dict(window=f"n={n} → {year}", method=mid, **r) for r in score_variants(named, zy[pxy], cy)]
time_aware = pd.concat([time_aware, pd.DataFrame(rows)], ignore_index=True)
time_aware.to_csv(f"{OUT}/{SITE}_time_aware.csv", index=False)
if YEAR_CHECK.get(SITE):
    display(time_aware[time_aware.window != f"n={N_LONG} → {TRUTH_YEAR}"].round(3))
else:
    print(f"no year check at {SITE}")

np.savez_compressed(f"{OUT}/{SITE}_pixels.npz", keep=keep[px], truth=t_true, change_class=cls, z_old=z_old_px,
                    **{f"{mid}_n{n}": z for (n, mid), z in EST.items()},
                    **{f"{mid}_{k}_n{N_LONG}".replace(" ", "_"): v for mid, d in TA.items() for k, v in d.items()},
                    **{f"{mid}_break_n{N_LONG}": k for mid, k in BREAKS.items()})
print(f"-> {OUT}/{SITE}_time_aware.csv, {OUT}/{SITE}_pixels.npz")

## 12 · Provenance

The commit and the files that differ from it, the package versions and the host.

In [ ]:
import platform, socket, subprocess
import scipy
git = lambda *a: subprocess.run(["git", *a], capture_output=True, text=True).stdout.rstrip()
dirty = [l for l in git("status", "--porcelain").splitlines() if not l.startswith("??")]
print("commit  ", git("rev-parse", "HEAD"), git("rev-parse", "--abbrev-ref", "HEAD"))
print(f"modified {len(dirty)} tracked file(s)" + "".join(f"\n  {l}" for l in dirty[:15]))
print("host    ", socket.gethostname(), "· OMP_NUM_THREADS", os.environ.get("OMP_NUM_THREADS"))
print("versions", f"python {platform.python_version()} · numpy {np.__version__} · pandas {pd.__version__} · "
      f"xarray {xr.__version__} · scipy {scipy.__version__} · pyintertidal {pit.__version__}")